In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm
import time, os, json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = "__4__baseline_vit_pretrained_cifar10.pth"
IMG_SIZE  = 32

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)


# ── Rebuild model architecture (must match training) ──────────
def build_model(num_classes=10):
    model = timm.create_model(
        'vit_small_patch16_224',
        pretrained=False,
        num_classes=num_classes,
        img_size=IMG_SIZE,
        patch_size=4,      # 8×8 grid = 64 patches
    )
    return model


model = build_model(10)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()
print(f"✓ Weights loaded from {SAVE_PATH}")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {total_params:,}")

# ── Test loader ───────────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128,
                                           shuffle=False, num_workers=0)

# ── Classification metrics ────────────────────────────────────
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        all_preds.extend(model(inputs).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall    = recall_score(all_labels, all_preds, average='macro')
f1        = f1_score(all_labels, all_preds, average='macro')

# ── FLOPs ─────────────────────────────────────────────────────
dummy_flops = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
macs, _     = thop_profile(model, inputs=(dummy_flops,), verbose=False)
flops_G     = macs * 2 / 1e9

# ── Parameters & disk size ────────────────────────────────────
size_mb = os.path.getsize(SAVE_PATH) / 1e6

# ── Inference time — CPU (run FIRST, before GPU timing) ──────
print("\n⏱  Measuring CPU inference times ...")
model_cpu = build_model(10)
model_cpu.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
model_cpu = model_cpu.eval()

dummy_single_cpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
dummy_batch_cpu  = torch.randn(128, 3, IMG_SIZE, IMG_SIZE)

# single image — CPU warmup + measure
with torch.no_grad():
    for _ in range(10):
        model_cpu(dummy_single_cpu)

times_cpu_single = []
with torch.no_grad():
    for _ in range(100):
        t0 = time.perf_counter()
        model_cpu(dummy_single_cpu)
        times_cpu_single.append(time.perf_counter() - t0)

inf_ms_single_cpu = np.mean(times_cpu_single) * 1000

# batch 128 — CPU warmup + measure
with torch.no_grad():
    for _ in range(5):
        model_cpu(dummy_batch_cpu)

times_cpu_batch = []
with torch.no_grad():
    for _ in range(20):
        t0 = time.perf_counter()
        model_cpu(dummy_batch_cpu)
        times_cpu_batch.append(time.perf_counter() - t0)

inf_ms_batch128_cpu     = np.mean(times_cpu_batch) * 1000
inf_ms_per_img_cpu      = inf_ms_batch128_cpu / 128
throughput_imgs_sec_cpu = 128 / (inf_ms_batch128_cpu / 1000)

del model_cpu, dummy_single_cpu, dummy_batch_cpu
print("✓ CPU timing done")

# ── Inference time — single image, GPU, SYNCHRONIZED ─────────
print("⏱  Measuring GPU inference times ...")
dummy_single_gpu = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

with torch.no_grad():
    for _ in range(50):
        model(dummy_single_gpu)
torch.cuda.synchronize()

times = []
with torch.no_grad():
    for _ in range(500):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy_single_gpu)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

inf_ms_single_gpu = np.mean(times) * 1000

# ── Inference time — batch 128, GPU, CUDA events ─────────────
dummy_batch_gpu = torch.randn(128, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
start_ev        = torch.cuda.Event(enable_timing=True)
end_ev          = torch.cuda.Event(enable_timing=True)

with torch.no_grad():
    for _ in range(10):
        model(dummy_batch_gpu)
torch.cuda.synchronize()

batch_times_gpu = []
with torch.no_grad():
    for _ in range(100):
        start_ev.record()
        model(dummy_batch_gpu)
        end_ev.record()
        torch.cuda.synchronize()
        batch_times_gpu.append(start_ev.elapsed_time(end_ev))

inf_ms_batch128_gpu     = np.mean(batch_times_gpu)
inf_ms_per_img_gpu      = inf_ms_batch128_gpu / 128
throughput_imgs_sec_gpu = 128 / (inf_ms_batch128_gpu / 1000)
print("✓ GPU timing done")

# ── Print results ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("BASELINE METRICS  (Pretrained ViT — 32×32, patch_size=4)")
print(f"{'='*55}")
print(f"  Accuracy          : {acc:.4f}")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")
print(f"  Parameters        : {total_params:,}")
print(f"  Model size        : {size_mb:.2f} MB")
print(f"  FLOPs             : {flops_G:.3f} GFLOPs  (at {IMG_SIZE}×{IMG_SIZE})")
print(f"  --- Inference (CPU) ---")
print(f"  Single image CPU  : {inf_ms_single_cpu:.3f} ms")
print(f"  Batch-128 CPU     : {inf_ms_batch128_cpu:.2f} ms")
print(f"  Per-image CPU     : {inf_ms_per_img_cpu:.3f} ms")
print(f"  Throughput CPU    : {throughput_imgs_sec_cpu:.1f} imgs/sec")
print(f"  --- Inference (GPU) ---")
print(f"  Single image GPU  : {inf_ms_single_gpu:.3f} ms  (synchronized)")
print(f"  Batch-128 GPU     : {inf_ms_batch128_gpu:.2f} ms  (CUDA events)")
print(f"  Per-image GPU     : {inf_ms_per_img_gpu:.3f} ms")
print(f"  Throughput GPU    : {throughput_imgs_sec_gpu:.1f} imgs/sec")

# ── Save JSON ─────────────────────────────────────────────────
baseline_metrics = {
    "model"     : "vit_small_patch4_32_pretrained",
    "accuracy"  : acc,
    "precision" : precision,
    "recall"    : recall,
    "f1"        : f1,
    "params"    : total_params,
    "size_mb"   : size_mb,
    "flops_G"   : flops_G,
    "input_resolution": IMG_SIZE,
    "patch_size": 4,
    "num_patches": 64,
    "inference_ms": {
        "cpu": {
            "single_img_ms"      : round(inf_ms_single_cpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_cpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_cpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_cpu, 1),
            "timing_method"      : "time.perf_counter()",
        },
        "gpu": {
            "single_img_ms"      : round(inf_ms_single_gpu, 4),
            "batch128_ms"        : round(inf_ms_batch128_gpu, 4),
            "per_img_ms"         : round(inf_ms_per_img_gpu, 4),
            "throughput_imgs_sec": round(throughput_imgs_sec_gpu, 1),
            "timing_method"      : "CUDA events + torch.cuda.synchronize()",
        },
    },
}
with open("__4__baseline_metrics_v3.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)
print(f"\n✓ Metrics saved → __4__baseline_metrics_v3.json")

e:\baseline_ViT\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Weights loaded from __4__baseline_vit_pretrained_cifar10.pth
  Parameters: 21,342,346


e:\baseline_ViT\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



⏱  Measuring CPU inference times ...
✓ CPU timing done
⏱  Measuring GPU inference times ...
✓ GPU timing done

BASELINE METRICS  (Pretrained ViT — 32×32, patch_size=4)
  Accuracy          : 0.9666
  Precision (macro) : 0.9667
  Recall    (macro) : 0.9666
  F1-score  (macro) : 0.9666
  Parameters        : 21,342,346
  Model size        : 85.43 MB
  FLOPs             : 2.763 GFLOPs  (at 32×32)
  --- Inference (CPU) ---
  Single image CPU  : 10.409 ms
  Batch-128 CPU     : 651.21 ms
  Per-image CPU     : 5.088 ms
  Throughput CPU    : 196.6 imgs/sec
  --- Inference (GPU) ---
  Single image GPU  : 4.304 ms  (synchronized)
  Batch-128 GPU     : 54.09 ms  (CUDA events)
  Per-image GPU     : 0.423 ms
  Throughput GPU    : 2366.5 imgs/sec

✓ Metrics saved → __4__baseline_metrics_v3.json
